# Figure 7: SLAP2 Glutamate Quality Control

Migrated from [Ido Aizenbud's community PR #171](https://github.com/AllenNeuralDynamics/openscope-community-predictive-processing/pull/171), commit `a0f6af8a05f296d1c376bd10e1edd70766699c25`.

The publication retains panel labels **C, G, and K** from the original multimodal plan. C shows both imaging planes and an example synapse; G shows signal/noise measurement and archive-wide quality classes; K shows context rates and class composition. Figure 7 currently contains these SLAP2 results; unfinished analyses for other modalities remain marked in the manuscript.

The committed tables contain **20 sessions, 8 mice, and 2,540 sources**. The final upstream revision uses one archive-wide class fit. This notebook runs offline from those tables and a checksummed four-trace example snapshot, and writes previews under `_build/slap2-qc-notebook/`.

The event-based scheme follows [discussion #156](https://github.com/AllenNeuralDynamics/openscope-community-predictive-processing/discussions/156):

1. Detect local maxima on matched-filtered ΔF/F at 3 noise SD, using a 20 ms exponential kernel.
2. Measure local peak-minus-baseline amplitudes on the raw trace in raw-noise SD; bins are <2, 2 to <4, and ≥4 SD.
3. Fit k-means (k=3, seed=0) to standardized event fractions and order the labels by the highest-amplitude fraction.

Analysis and rendering live in `src/openscope_p3_publication/`; deliberate cloud refreshes use `scripts/extract_slap2_glutamate_qc.py`. See [the data guide](../data/slap2-glutamate-qc/README.md) for provenance and refresh commands.

### Imports

In [ ]:
import pandas as pd
from IPython.display import Image, display

from openscope_p3_publication import slap2_glutamate_figure7_panels as panels
from openscope_p3_publication import slap2_glutamate_qc_events as qc

REPO = panels.REPO_ROOT
DATA_DIR = panels.DATA_DIR
OUT_DIR = REPO / "_build" / "slap2-qc-notebook"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Matched-filter decay constant:", qc.TAU_IGLUSNFR4F_S, "s")
print("Detection threshold:", qc.SD_DETECT, "SD of filtered-trace noise")
print("Large-event boundary:", qc.SD_LARGE, "SD of raw-trace noise")

### The Example Session

The example is `sub-794237_ses-20250508T145040` from DANDI 001424. The committed 13.05 MiB lossless snapshot retains both planes' activity/mean images, all 179 source masks, and four selected full-session traces with exact timestamps. It avoids downloading the 2.76 GB primary NWB during notebook execution.

Extraction checks the selected traces' event counts, noise estimates, and median event amplitudes against the original tables. Source identifiers, observed DANDI digests, and the snapshot checksum are recorded in the adjacent provenance JSON.

In [ ]:
EXAMPLE_PATH = panels.EXAMPLE_PATH
sess = panels.load_example_snapshot(EXAMPLE_PATH)
print("Example session:", sess["session_id"])
print("Snapshot size:", round(EXAMPLE_PATH.stat().st_size / 1024**2, 2), "MiB")

### Step 1: Original Example-Session Metrics

Load the committed `slap2_event_metrics.csv`, which contains one row per source and the original within-session classifications. The complete per-event table is not committed and is not required for these panels.

A full-session analysis remains available through `python -m openscope_p3_publication.slap2_glutamate_qc_events --nwb <local-file> --output <staging-directory>`. This is an explicit refresh operation, not part of the notebook's default execution.

In [ ]:
metrics = pd.read_csv(DATA_DIR / "slap2_event_metrics.csv")
print("Loaded original example-session metrics:", len(metrics), "sources")

### What the metrics mean

| metric | meaning |
|---|---|
| `event_rate_hz` | detected events per second of valid recording |
| `false_pos_frac` | downward peaks of the matched-filtered trace beyond the same 3 SD floor, divided by detections. Release only pushes the trace up, so downward crossings are an **empirical noise null** |
| `robust_snr` | `(P95(ΔF/F) − P50(ΔF/F)) / noise SD` — the portable 95/50 metric proposed in #156 |
| `frac_events_lt2sd`, `frac_events_2_4sd`, `frac_events_gt4sd` | fraction of the synapse's events with raw amplitude < 2, 2–4, > 4 SD of the raw-trace noise — the three features the classes are fitted on |
| `quality_class` | Low SNR / Intermediate / High SNR (k-means on the three fractions); `Excluded (no baseline noise)` for rectified traces that sit at exactly zero for more than half the session |

In [ ]:
cols = [
    "event_rate_hz",
    "false_pos_frac",
    "robust_snr",
    "frac_events_lt2sd",
    "frac_events_2_4sd",
    "frac_events_gt4sd",
]
print(f"{len(metrics)} synapses, {int(metrics.n_events.sum()):,} events\n")
print("Median metric by quality class (fitted within this session):")
display(metrics.groupby("quality_class")[cols].median().round(3))
print("Class composition by dendritic compartment (row %):")
display((pd.crosstab(metrics.dmd, metrics.quality_class, normalize="index") * 100).round(1))

### Step 2: Archive-Wide Tables

The committed files are:

| File | Contents |
|---|---|
| `slap2_metrics_all_sessions.csv` | 2,540 sources, with within-session and archive-wide quality labels |
| `slap2_context_rates_all_sessions.csv` | Event-bearing source/context records, event counts, valid recorded seconds, and rates |
| `slap2_session_summary.csv` | One row per session |

The complete archive analysis is an explicit cloud refresh:

```bash
uv run --extra nwb-view python scripts/extract_slap2_glutamate_qc.py \
  --output /tmp/slap2-qc-refresh --cache /tmp/slap2-nwb-cache
```

Run refreshes in staging, review changes, and regenerate dependent provenance and figures together. The final source revision uses **one fit over all 2,521 classified sources**, not separate per-cohort fits. Nineteen sources have no assigned class.

The 12 glutamate-only sessions contain two-sided ΔF/F; the 8 glutamate-plus-calcium sessions contain non-negative, denoised ΔF/F with a different noise scale. Cross-cohort class differences must not be interpreted as preparation-quality differences.

In [ ]:
m_all, ctx_all, summary = panels.publication_tables(DATA_DIR)
print(f"{len(summary)} sessions, {summary.subject.nunique()} mice, {len(m_all):,} synapses")
display(
    summary[
        [
            "subject",
            "session",
            "layout",
            "has_calcium",
            "n_sources",
            "duration_s",
            "median_event_rate_hz",
            "median_false_pos_frac",
            "median_robust_snr",
            "n_low_snr",
            "n_intermediate",
            "n_high_snr",
        ]
    ]
)

### Step 3: Render the Panels Offline

The selected session arrays have already been loaded from the committed, checksummed example. Rendering writes deterministic SVG and PNG previews into the ignored `OUT_DIR`; no primary NWB or cloud access is needed.

The categorical versions preserve the original ordinal palette and C/G/K identifiers. The optional continuous-color C/G versions are retained below. The publication's standard build calls the same rendering functions and combines C/G/K into Figure 7.

In [ ]:
panels.set_style()
m_ex = m_all[m_all.session == sess["session_id"]].reset_index(drop=True)
print(
    f"Example session: {len(m_ex)} sources, classes {m_ex.quality_class.value_counts().to_dict()}"
)

#### Panel C — example projection, single synapse, trace with detected events

Both imaging planes are shown: recording proximal (DMD1) and apical (DMD2) dendrites of the same neuron
simultaneously is what defines the SLAP2 prep. SLAP2 samples only user-selected regions, so each plane is
a set of disjoint imaged islands (dashed boxes, with their synapse counts). Footprints are coloured by
quality class; the circled synapse is the one whose trace is shown below.

In [ ]:
panels.render_panel_c(sess, m_ex, OUT_DIR)
display(Image(str(OUT_DIR / "figure7_panelC_slap2_glutamate.png")))

#### Panel G — measuring signal and noise, and the class definition

Top: the two steps of the measurement on 5 s of the same synapse — detection on the matched-filtered
trace (3 SD), amplitude on the raw trace (brackets, coloured by amplitude bin).
Bottom left: how a synapse gets its class — the amplitude histogram of one representative synapse per
class with its < 2 / 2–4 / > 4 SD fractions. Bottom right: every classified synapse of the archive in
that feature space, coloured by the archive-wide class (rings: the three examples).

In [ ]:
panels.render_panel_g(sess, m_ex, m_all, summary, OUT_DIR)
display(Image(str(OUT_DIR / "figure7_panelG_slap2_glutamate.png")))

#### Panel K — class distributions across context, compartment, and neurons

Top: event rate per synapse by stimulus context and class, one axis per cohort because their stimulus
vocabularies differ (the glutamate-only sessions ran the standard oddball paradigm, with contexts inferred
from the orientation statistics; the glutamate + calcium sessions ship one interval table per named block);
a class with fewer than 10 synapses in a cohort gets no bars.
Bottom left: class composition by dendritic compartment. Bottom right: class composition of each
neuron — one per session — in acquisition order, grouped by mouse.

In [ ]:
panels.render_panel_k(m_all, ctx_all, OUT_DIR)
display(Image(str(OUT_DIR / "figure7_panelK_slap2_glutamate.png")))

#### Variant — one continuous scale instead of the three classes

The classes are cuts through a continuum, and C and G are the two panels that can carry a colourbar
instead of a legend: an ROI map and a scatter. `color_by` renders them that way, writing
`figure7_panel{C,G}_slap2_glutamate_gradient.svg/.png` next to the defaults; the class versions above
are unchanged. Panel K keeps the classes — a stacked or grouped bar needs a categorical variable, and
the three-class scheme is what keeps this column comparable to the mesoscope column of #156.

The scalar is `median_event_raw_sd`, the synapse's median event amplitude in SD of its own noise: the
same unit as the rest of panel G, the centre of mass of the histograms in G-ii, and spearman 0.99 with
the > 4 SD fraction the classes are ordered by. `robust_snr` and `frac_events_gt4sd` are also accepted.
`robust_snr` is the obvious "SNR" but a poor gradient — its class medians (2.32 / 2.65 / 2.70
archive-wide) overlap almost completely, and every synapse above 10 is in the glutamate + calcium
cohort, whose non-negative ΔF/F shrinks σ.

Colour scales are per panel, not shared: C spans this session's synapses (2–98th percentile), G spans
the archive. So the glutamate + calcium cohort reading warmer in G is that same ΔF/F offset, not a
prep difference. ROI outlines in C get a dark halo, because they sit on bright puncta separated by
black background and a single stroke colour cannot stay legible on both.

In [ ]:
COLOR_BY = "median_event_raw_sd"  # or "robust_snr", "frac_events_gt4sd"

panels.render_panel_c(sess, m_ex, OUT_DIR, color_by=COLOR_BY)
panels.render_panel_g(sess, m_ex, m_all, summary, OUT_DIR, color_by=COLOR_BY)
for p in ("C", "G"):
    display(Image(str(OUT_DIR / f"figure7_panel{p}_slap2_glutamate_gradient.png")))

In [ ]:
print("Wrote:")
for f in sorted(OUT_DIR.glob("figure7_panel*")):
    print(f"  {f}  ({f.stat().st_size / 1024:.0f} KB)")

## Reading the Panels

**Signal quality and class definition.** The snapshot contains 2,540 sources across 20 sessions from 8 mice. One archive-wide fit labels 665 Low SNR, 1,026 Intermediate, and 830 High SNR; 19 sources have undefined noise or event-amplitude features and remain unclassified. The example session contains 179 sources. The classes describe recorded signal properties, not molecular identities or a common preparation-quality scale across processing cohorts.

**Context rates.** The imported table contains only event-bearing source/context pairs. Panel K averages retained records with at least 5 valid recorded seconds; it does not insert absent zero-event pairs. Means and SEM are therefore conditional on those records. Classes with fewer than 10 sources in a cohort and named contexts with fewer than 100 retained records are not drawn.

**Compartments and sessions.** DMD1 and DMD2 represent proximal and apical sampling, respectively. Composition plots use classified sources and group recording sessions by mouse. They do not establish longitudinal identity across sessions.

**Source limitations.** Contexts in the older glutamate-only sessions are inferred from orientation statistics, whereas dual-channel sessions use named interval tables. The latter contain non-negative, denoised traces with a different noise scale; differences between cohorts are not evidence of preparation quality. The original source did not record its NWB digests. Migration provenance records the historical table hashes separately from the asset metadata observed during migration, and verifies the four example traces against their source metrics without rerunning the full archive.